### Bigramas estadisiticos vs Bigramas de Redes Neuronales

In [1]:
# Abrimos el archivo de nombres de ciudades y lo leemos completo.
# .read() devuelve TODO el texto del archivo; .split('\n') lo corta en una lista,
# un nombre de ciudad por cada línea.
dataset = open('data/city_names_puny.txt', 'r').read().split('\n')
dataset  # El archivo viene en "punycode": una codificación ASCII para representar texto no-ASCII (acentos, otros alfabetos)

# Separamos los nombres que están en punycode (empiezan con 'xn--') de los que no.
puny = [x for x in dataset if 'xn--' in x]        # nombres codificados en punycode
nopuny = [x for x in dataset if 'xn--' not in x]  # nombres "normales" (ASCII simple)

In [2]:
import numpy as np

In [3]:
import idna  # Librería para codificar/decodificar punycode (IDNA = nombres de dominio internacionalizados)

# def utf8_to_punycode(text: str) -> str:
#     """Encodes a UTF-8 string to its Punycode representation."""
#     return idna.encode(text).decode('ascii')

def punyencode(text: str) -> str:
    """Codifica un string UTF-8 a punycode, manejando los espacios codificando cada palabra por separado."""
    # Dividimos el texto por espacios, codificamos cada palabra a punycode (que son bytes),
    # la pasamos a texto ASCII con .decode('ascii'), y volvemos a unir todo con espacios.
    return " ".join([idna.encode(word).decode('ascii') for word in text.split()])

def punydecode(punycode: str) -> str:
    """Decodifica un string en punycode de vuelta a UTF-8 (operación inversa de punyencode)."""
    #return idna.decode(punycode)
    return " ".join([idna.decode(word) for word in punycode.split()])

def process_name(name):
    name = name.lower()          # Pasamos todo a minúsculas para unificar (así 'A' y 'a' son lo mismo)
    for n in name.split():       # Recorremos cada palabra del nombre
        if len(n) < 2:           # idna no admite palabras de menos de 2 caracteres -> descartamos el nombre
            return ''
    try:
        return punyencode(name)  # Intentamos codificar el nombre a punycode
    except:
        # Si la codificación falla por cualquier motivo, devolvemos string vacío para descartarlo
        #print(f'Cant convert {name}')
        return ''

In [4]:
# Ahora agregamos '*' al PRINCIPIO y al FINAL de cada nombre.
# Así el modelo también aprende:
#   - qué caracter suele INICIAR una palabra  (bigrama  *  -> algo)
#   - qué caracter suele TERMINARLA           (bigrama algo ->  * )
for d in dataset[:3]:                          # Primeros 3 nombres
    print('\n')
    example = '*' + d + '*'                     # Ej: '*kotamangalam*'
    for c1, c2 in zip(example, example[1:]):
        print(c1, c2)

# Creamos el "vocabulario" ORDENADO: '*' primero y luego el resto de caracteres ordenados alfabéticamente.
charset = ['*'] + sorted(list(set([y for x in dataset for y in x])))

# ctoi = "char to index": mapea cada caracter a un número (su posición). Ej: {'*': 0, 'a': 1, 'b': 2, ...}
ctoi = {c: i for i, c in enumerate(charset)}
# itoc = "index to char": el mapeo INVERSO, de número a caracter. Ej: {0: '*', 1: 'a', 2: 'b', ...}
itoc = {i: c for i, c in enumerate(charset)}



* h
h e
e i
i s
s t
t -
- o
o p
p -
- d
d e
e n
n -
- b
b e
e r
r g
g *


* h
h a
a r
r c
c h
h i
i e
e s
s *


* f
f l
l o
o r
r e
e n
n n
n e
e s
s *


### Contando Bigramas
Acá contamos los bigramas a mano y los metemos en una matriz

In [5]:
# Creamos una matriz de CEROS de tamaño (vocabulario x vocabulario), es decir 40x40.
# La fila i y la columna j representarán el bigrama (caracter i -> caracter j).
# .astype(int) la convierte a enteros, porque los conteos son números enteros.
bigram_count = np.zeros((len(charset), len(charset))).astype(int)
for d in dataset:  # <== remove [:10] to run for all examples
    example = ['*'] + list(d) + ['*']
    for c1, c2 in zip(example, example[1:]):
        bigram_count[ctoi[c1], ctoi[c2]] += 1
bigram_count += 1
bigram_count


array([[    1,     1,     1, ..., 23497,  1143,  1368],
       [    1,     1,     1, ...,  6163,   219,   258],
       [    1,     1, 29158, ...,  1112,   936,  1074],
       ...,
       [  477,    55,   228, ...,    34,    39,    19],
       [ 5020,   842,  1160, ...,    36,   321,    84],
       [  973,   322,   782, ...,    31,   381,   502]], shape=(40, 40))

In [6]:
p = bigram_count[0]/sum(bigram_count[0])
p

array([6.95004309e-06, 6.95004309e-06, 6.95004309e-06, 6.95004309e-06,
       6.95004309e-06, 3.47502155e-05, 4.17002585e-05, 1.39000862e-05,
       6.95004309e-06, 6.95004309e-06, 1.39000862e-05, 1.39000862e-05,
       6.95004309e-06, 6.95004309e-06, 4.63915376e-02, 6.90278280e-02,
       6.63173112e-02, 2.65005143e-02, 2.35467460e-02, 1.96825220e-02,
       3.21786995e-02, 2.92388313e-02, 1.15301215e-02, 1.03138639e-02,
       3.76831336e-02, 5.23268744e-02, 6.29048400e-02, 2.90859303e-02,
       1.89527675e-02, 5.05615635e-02, 4.10052542e-03, 2.67924161e-02,
       1.02749437e-01, 4.04214506e-02, 7.35314559e-03, 2.63754135e-02,
       2.50271052e-02, 1.63305162e-01, 7.94389925e-03, 9.50765895e-03])

In [7]:
bigram_prob = bigram_count / bigram_count.sum(axis=1, keepdims=True)
bigram_prob

array([[6.95004309e-06, 6.95004309e-06, 6.95004309e-06, ...,
        1.63305162e-01, 7.94389925e-03, 9.50765895e-03],
       [2.07344129e-05, 2.07344129e-05, 2.07344129e-05, ...,
        1.27786187e-01, 4.54083643e-03, 5.34947853e-03],
       [1.02867959e-05, 1.02867959e-05, 2.99942394e-01, ...,
        1.14389170e-02, 9.62844093e-03, 1.10480188e-02],
       ...,
       [1.39989435e-02, 1.61413394e-03, 6.69131889e-03, ...,
        9.97828256e-04, 1.14456771e-03, 5.57609908e-04],
       [2.37128011e-01, 3.97732641e-02, 5.47945205e-02, ...,
        1.70051960e-03, 1.51629665e-02, 3.96787907e-03],
       [6.45782173e-02, 2.13712086e-02, 5.19015066e-02, ...,
        2.05747660e-03, 2.52870512e-02, 3.33178470e-02]], shape=(40, 40))

In [8]:
ll = 0
i = 0
for ciudad in dataset:
    for c1,c2 in zip(ciudad, ciudad[1:]):
        ll += -np.log(bigram_prob[ctoi[c1]][ctoi[c2]])
        i += 1
ll/i
        

np.float64(2.752153426674224)

### El modelo de redes neuronales

In [9]:
import torch
import torch.nn.functional as F  # Funciones auxiliares de PyTorch; acá usamos one_hot

In [10]:
nclasses = len(charset)

In [11]:
# --- CELDA DE INICIALIZACION: correr UNA sola vez (o cuando quieras empezar de cero) ---

# Generador con semilla fija (42): los pesos iniciales son siempre los mismos
# y los resultados reproducibles.
g = torch.Generator().manual_seed(42)

# W = matriz de pesos (40, 40):
#   - 40 filas    -> un peso por cada entrada posible (el caracter actual, en one-hot)
#   - 40 columnas -> una neurona por cada caracter que podemos predecir
# requires_grad=True -> PyTorch calculara dloss/dW (lo necesitamos para entrenar).
W = torch.randn((nclasses, nclasses), generator=g, requires_grad=True)

# Preparamos los datos para el enfoque de RED NEURONAL (en lugar de solo contar).
# xs = entradas  (caracter actual)    -> como INDICES numericos
# ys = objetivos (caracter siguiente) -> como INDICES numericos
xs, ys = [], []
for d in dataset:
    example = ['*'] + list(d) + ['*']
    for c1, c2 in zip(example, example[1:]):
        xs.append(ctoi[c1])                 # indice del caracter actual
        ys.append(ctoi[c2])                 # indice del caracter siguiente
xs = torch.tensor(xs)                       # de lista de Python a tensor de PyTorch
ys = torch.tensor(ys)

nsamples = xs.nelement()

# El one-hot de las entradas no cambia nunca: lo calculamos UNA vez, fuera del loop.
xenc = F.one_hot(xs, num_classes=nclasses).float()

print('cantidad de bigramas:', nsamples)
print('xenc:', tuple(xenc.shape), ' W:', tuple(W.shape))

cantidad de bigramas: 1737335
xenc: (1737335, 40)  W: (40, 40)


In [21]:
# --- CELDA DE ENTRENAMIENTO: se puede correr VARIAS veces seguidas ---
# W ya existe (lo creo la celda de arriba), asi que cada ejecucion CONTINUA el
# entrenamiento desde donde quedo en vez de empezar de cero.
# Para reiniciar: volver a correr la celda de inicializacion.

iterations = 100
lr = 50

for k in range(iterations):
    # --- forward pass ---
    logits = xenc @ W                                    # log(counts) predichos: (n, 40)
    counts = logits.exp()                                # exponenciamos -> "cuentas"
    probs = counts / counts.sum(axis=1, keepdims=True)   # normalizamos -> softmax
    # NLL promedio sobre la etiqueta correcta, + regularizacion L2 que empuja W hacia 0.
    # Ojo: (W**2) es una matriz (40, 40); hace falta .mean() para que la loss sea un ESCALAR,
    # si no backward() falla ("grad can be implicitly created only for scalar outputs").
    loss = -probs[range(nsamples), ys].log().mean() + 0.001 * (W**2).mean()

    if k % 10 == 0:
        print(f'{k:4d}  {loss.item()}')

    # --- backward pass ---
    W.grad = None                                        # limpiamos: PyTorch ACUMULA gradientes
    loss.backward()                                      # calcula dloss/dW
    # --- update ---
    W.data += -lr * W.grad                               # movemos W en contra del gradiente
    # Usamos W.data (modificacion in-place) para no crear un tensor nuevo:
    # asi W sobrevive entre ejecuciones de la celda conservando lo aprendido.

print()
print('loss final             :', round(loss.item(), 4))
print('baseline al azar       :', round(float(-np.log(1 / nclasses)), 4), '= -log(1/40)')
print('modelo de conteo       : ~2.75  <- el techo de este modelo: la red aprende la MISMA tabla')

   0  2.7294774055480957
  10  2.7292473316192627
  20  2.72902774810791
  30  2.7288191318511963
  40  2.7286198139190674
  50  2.7284300327301025
  60  2.728248357772827
  70  2.728074789047241
  80  2.727908134460449
  90  2.7277493476867676

loss final             : 2.7276
baseline al azar       : 3.6889 = -log(1/40)
modelo de conteo       : ~2.75  <- el techo de este modelo: la red aprende la MISMA tabla


La salida del modelo es la log count

In [13]:
bigram_prob[ctoi['l'], ctoi['a']]

np.float64(0.20709693072910348)

In [19]:
nnprob = W.exp()/W.exp().sum(axis=1, keepdims=True)
nnprob

tensor([[0.0004, 0.0004, 0.0004,  ..., 0.1631, 0.0077, 0.0093],
        [0.0008, 0.0013, 0.0010,  ..., 0.1273, 0.0032, 0.0047],
        [0.0008, 0.0008, 0.2999,  ..., 0.0114, 0.0096, 0.0110],
        ...,
        [0.0132, 0.0007, 0.0057,  ..., 0.0007, 0.0013, 0.0019],
        [0.2364, 0.0387, 0.0538,  ..., 0.0031, 0.0137, 0.0043],
        [0.0631, 0.0166, 0.0503,  ..., 0.0054, 0.0235, 0.0308]],
       grad_fn=<DivBackward0>)

In [20]:
torch.allclose(torch.tensor(bigram_prob, dtype=float), nnprob.to(float))

False